# Assignment 2: Gradio dashboard


# Olive peacock spot: decision dashboard

Gradio dashboard for the cooperative's treatment decision. It loads the finetuned
model from `weights/` only to classify uploaded photos, and reads everything else
(test-set probabilities, cost settings, chosen thresholds, per-class results) from
`results/`, written by `train.ipynb`. Nothing is trained or re-evaluated here.

In [1]:
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")   # render figures off-screen: Gradio runs callbacks in worker threads
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import gradio as gr
from ultralytics import YOLO

ROOT = Path.cwd()
RESULTS = ROOT / "results"
model = YOLO(str(ROOT / "weights" / "olive_cls_best.pt"))
CLASS_NAMES = [model.names[i] for i in range(len(model.names))]
POSITIVE = "olive_peacock_spot"

# Everything below was produced by train.ipynb
preds = pd.read_csv(RESULTS / "test_predictions.csv")
operating = json.loads((RESULTS / "operating_point.json").read_text())
review = json.loads((RESULTS / "review_band.json").read_text())
train_config = json.loads((RESULTS / "train_config.json").read_text())
per_class = pd.read_csv(RESULTS / "per_class_finetuned.csv", index_col=0)
confusion = pd.read_csv(RESULTS / "confusion_finetuned.csv", index_col=0)
baselines = pd.read_csv(RESULTS / "baselines_comparison.csv", index_col=0)

COSTS = operating["costs"]            # €/ha for TP, FP, FN, TN
BASE_RATE = operating["base_rate"]
CHOSEN_T = operating["threshold"]     # best single threshold (model alone)
C_REVIEW = review["review_cost"]
CASES_PER_DAY = review["cases_per_day"]
REVIEW_MINUTES = review["review_minutes"]
IMGSZ = train_config["imgsz"]

scores = preds[f"p_{POSITIVE}"].to_numpy()
is_pos = (preds["true"] == POSITIVE).to_numpy()
print(f"Loaded {len(preds)} test predictions; chosen threshold {CHOSEN_T}, "
      f"review band {review['t_low']} to {review['t_high']}")

/Users/joseluishervasdecela_1/Desktop/daiist-assignment-2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 323 test predictions; chosen threshold 0.39, review band 0.005 to 0.86


In [2]:
OUTCOMES = ["not treated", "sent to review", "treated"]

def outcome_index(s, t_low, t_high):
    # 0 = not treated, 1 = sent to review, 2 = treated
    t_high = max(t_high, t_low)          # if the sliders cross, there is no review band
    return np.where(s < t_low, 0, np.where(s >= t_high, 2, 1))

def evaluate_band(t_low, t_high, base_rate):
    out = outcome_index(scores, t_low, t_high)
    counts = pd.DataFrame(
        [[int((out[is_pos] == k).sum()) for k in range(3)],
         [int((out[~is_pos] == k).sum()) for k in range(3)]],
        index=["true peacock spot", "true healthy or Aculus"], columns=OUTCOMES)
    pos = [(out[is_pos] == k).mean() for k in range(3)]
    neg = [(out[~is_pos] == k).mean() for k in range(3)]
    # Same cost model as train.ipynb; a reviewed case gets the right decision
    cost_pos = pos[0] * COSTS["FN"] + pos[1] * (COSTS["TP"] + C_REVIEW) + pos[2] * COSTS["TP"]
    cost_neg = neg[0] * COSTS["TN"] + neg[1] * (COSTS["TN"] + C_REVIEW) + neg[2] * COSTS["FP"]
    cost = 1000 * (base_rate * cost_pos + (1 - base_rate) * cost_neg)
    share = base_rate * pos[1] + (1 - base_rate) * neg[1]
    missed = 1000 * base_rate * pos[0]
    return counts, cost, share, missed

# Sanity check: must reproduce the numbers train.ipynb saved
_, c, s, m = evaluate_band(review["t_low"], review["t_high"], BASE_RATE)
print(f"App: €{c:,.0f} per 1,000, review {s:.1%}, missed {m:.1f}  |  "
      f"train: €{review['cost_per_1000']:,.0f}, review {review['review_share']:.1%}")
_, c_single, _, _ = evaluate_band(CHOSEN_T, CHOSEN_T, BASE_RATE)
print(f"Single threshold {CHOSEN_T}: €{c_single:,.0f} (train: €{operating['cost_per_1000']:,.0f})")

App: €27,701 per 1,000, review 13.4%, missed 0.0  |  train: €27,701, review 13.4%
Single threshold 0.39: €28,947 (train: €28,947)


In [ ]:
# Visual identity: the three decisions get the colours of the grove.
# Leaf green = leave the parcel alone, lesion amber = ask the agronomist,
# copper blue = spray (Bordeaux mixture is that blue).
LEAF, AMBER, COPPER = "#4F6B2E", "#C08A1E", "#1F7A8C"
INK, MUTED, PAPER = "#1E2416", "#6B7260", "#F3F4EC"
OUTCOME_COLORS = [LEAF, AMBER, COPPER]
CLASS_LABELS = {"Healthy": "Healthy", "aculus_olearius": "Aculus olearius",
                "olive_peacock_spot": "Peacock spot"}

plt.rcParams.update({"font.size": 10, "axes.edgecolor": MUTED, "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.spines.top": False, "axes.spines.right": False})

def decision_strip(t_low, t_high):
    # Every test leaf as a dot on its P(peacock spot), one row per true class.
    # Logit scale: most scores sit near 0 or 1, and this spreads both ends out.
    t_high = max(t_high, t_low)
    eps = 1e-4
    lo, hi = max(t_low, eps), min(max(t_high, eps), 1 - eps)
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.axvspan(eps, lo, color=LEAF, alpha=0.12, lw=0)
    ax.axvspan(lo, hi, color=AMBER, alpha=0.18, lw=0)
    ax.axvspan(hi, 1 - eps, color=COPPER, alpha=0.12, lw=0)
    rng = np.random.default_rng(0)
    for row, cls in enumerate(CLASS_NAMES):
        s = np.clip(preds.loc[preds["true"] == cls, f"p_{POSITIVE}"].to_numpy(), eps, 1 - eps)
        y = row + rng.uniform(-0.28, 0.28, len(s))
        filled = cls == POSITIVE
        ax.scatter(s, y, s=16, linewidths=0.8,
                   facecolors=INK if filled else "none", edgecolors=INK, alpha=0.75)
    ax.set_xscale("logit")
    ax.set_xlim(eps, 1 - eps)
    ticks = [0.001, 0.01, 0.1, 0.5, 0.9, 0.99, 0.999]
    ax.set_xticks(ticks, [f"{t:g}" for t in ticks])
    ax.minorticks_off()
    ax.set_yticks(range(len(CLASS_NAMES)), [CLASS_LABELS[c] for c in CLASS_NAMES])
    ax.set_ylim(-0.6, len(CLASS_NAMES) - 0.4)
    ax.set_xlabel("P(peacock spot), logit scale")
    for x, txt, col in [(np.sqrt(eps * lo), "not treated", LEAF),
                        (np.sqrt(lo * hi), "review", AMBER),
                        (np.sqrt(hi * (1 - eps)) if hi < 0.5 else 1 - np.sqrt(eps * (1 - hi)), "treated", COPPER)]:
        ax.text(x, len(CLASS_NAMES) - 0.45, txt, color=col, ha="center", va="bottom", fontsize=9, weight="bold")
    fig.tight_layout()
    plt.close(fig)
    return fig

def cost_curve(t_low, t_high, base_rate):
    ts = np.round(np.arange(0, 1.0001, 0.01), 2)
    costs = [evaluate_band(t, t, base_rate)[1] for t in ts]
    _, band_cost_now, _, _ = evaluate_band(t_low, t_high, base_rate)
    chosen_cost = evaluate_band(CHOSEN_T, CHOSEN_T, base_rate)[1]
    do_nothing = 1000 * base_rate * COSTS["FN"]
    treat_all = 1000 * (base_rate * COSTS["TP"] + (1 - base_rate) * COSTS["FP"])
    fig, ax = plt.subplots(figsize=(9, 4.2))
    ax.plot(ts, costs, color=INK, lw=1.8, label="model alone, one threshold")
    ax.axhline(do_nothing, color=LEAF, ls="--", lw=1, label=f"never treat (€{do_nothing:,.0f})")
    ax.axhline(treat_all, color=COPPER, ls="--", lw=1, label=f"treat every parcel (€{treat_all:,.0f})")
    ax.axhline(band_cost_now, color=AMBER, lw=1.5, label=f"current review band (€{band_cost_now:,.0f})")
    ax.scatter([CHOSEN_T], [chosen_cost], color=INK, s=50, zorder=5,
               label=f"chosen threshold {CHOSEN_T:.2f} (€{chosen_cost:,.0f})")
    ax.set_xlabel("Threshold on P(peacock spot)")
    ax.set_ylabel("Cost per 1,000 parcels (€)")
    ax.set_ylim(0, max(do_nothing, treat_all) * 1.08)
    ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v/1000:.0f}k"))
    fig.legend(fontsize=8, loc="lower center", ncols=3, frameon=False)
    fig.tight_layout(rect=(0, 0.12, 1, 1))   # leave room for the legend under the plot
    plt.close(fig)
    return fig

def summary_html(t_low, t_high, base_rate):
    _, cost, share, missed = evaluate_band(t_low, t_high, base_rate)
    _, alone, _, _ = evaluate_band(CHOSEN_T, CHOSEN_T, base_rate)
    photos = share * CASES_PER_DAY
    minutes = photos * REVIEW_MINUTES
    # The agronomist's daily review time, as assumed in train.ipynb (1 hour)
    capacity_min = review["max_review_share"] * CASES_PER_DAY * REVIEW_MINUTES
    over = minutes > capacity_min
    review_label = f"sent to the agronomist: {photos:.0f} photos, {minutes:.0f} min a day"
    if over:
        review_label += f", over the {capacity_min:.0f} min available"
    stats = [
        (f"€{cost:,.0f}", "expected cost per 1,000 parcels", ""),
        (f"{share:.1%}", review_label, "over" if over else ""),
        (f"{missed:.1f}", "peacock spots missed per 1,000 parcels", ""),
        (f"€{abs(alone - cost):,.0f}",
         f"{'saved' if alone >= cost else 'more than'} the model alone at {CHOSEN_T:.2f}"
         .replace("saved the", "saved vs the"), ""),
    ]
    cells = "".join(f'<div class="stat {cls}"><span class="value">{v}</span>'
                    f'<span class="label">{l}</span></div>' for v, l, cls in stats)
    return f'<div class="stats">{cells}</div>'

def classify(image, t_low, t_high):
    if image is None:
        return None, '<div class="verdict empty">Upload a photo of one olive leaf on a white card.</div>'
    probs = model.predict(image, imgsz=IMGSZ, verbose=False)[0].probs.data.cpu().numpy()
    p = float(probs[CLASS_NAMES.index(POSITIVE)])
    k = int(outcome_index(np.array([p]), t_low, t_high)[0])
    advice = ["No copper needed for this parcel.",
              "Send this photo to the agronomist before deciding.",
              "Treat this parcel with copper."][k]
    verdict = (f'<div class="verdict" style="--tone:{OUTCOME_COLORS[k]}">'
               f'<span class="decision">{OUTCOMES[k].capitalize()}</span>'
               f'<span class="detail">P(peacock spot) = {p:.3f}. {advice}</span></div>')
    return {CLASS_LABELS[c]: float(v) for c, v in zip(CLASS_NAMES, probs)}, verdict

In [ ]:
THEME = gr.themes.Base(
    font=[gr.themes.GoogleFont("Archivo"), "system-ui", "sans-serif"],
    primary_hue=gr.themes.colors.emerald, neutral_hue=gr.themes.colors.stone,
).set(
    body_background_fill=PAPER, block_background_fill="white",
    block_border_color="#DADDCF", block_radius="4px", block_label_text_color=MUTED,
    slider_color=INK, button_primary_background_fill=COPPER,
    button_primary_background_fill_hover="#18626F", button_primary_text_color="white",
)

CSS = f"""
.gradio-container {{ max-width: 1100px !important; margin: 0 auto; }}
#title h1 {{ font-size: 2.1rem; font-weight: 800; letter-spacing: -0.02em; color: {INK}; margin-bottom: 0.2rem; }}
#title p {{ color: {MUTED}; max-width: 70ch; margin-top: 0; }}
.stats {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 1px; background: #DADDCF;
          border: 1px solid #DADDCF; }}
.stat {{ background: white; padding: 14px 16px; display: flex; flex-direction: column; gap: 4px; }}
.stat .value {{ font-size: 1.6rem; font-weight: 700; color: {INK}; font-variant-numeric: tabular-nums; }}
.stat .label {{ font-size: 0.85rem; color: {MUTED}; line-height: 1.3; }}
.stat.over .value, .stat.over .label {{ color: #9B2C2C; }}
.verdict {{ border-left: 6px solid var(--tone); background: white; padding: 18px 20px; }}
.verdict .decision {{ display: block; font-size: 1.8rem; font-weight: 800; color: var(--tone); }}
.verdict .detail {{ color: {INK}; }}
.verdict.empty {{ border-left-color: #DADDCF; color: {MUTED}; }}
@media (max-width: 700px) {{ .stats {{ grid-template-columns: repeat(2, 1fr); }} }}
"""

ASSUMPTIONS = f"""
**Costs per hectare.** Treating costs €{COSTS['TP']:.0f} whether or not the parcel is infected.
Missing peacock spot costs €{COSTS['FN']:.0f}, about 10% of a €4,000/ha harvest.
A review costs €{C_REVIEW:.2f} ({REVIEW_MINUTES} minutes of an agronomist at about €30/h),
and the agronomist is assumed to decide correctly.

**Volume.** About {CASES_PER_DAY} parcels a day in season; the agronomist can spend one hour
a day on reviews.

**Where the numbers come from.** Every figure on this page is computed from the model's
probabilities on the {len(preds)} test images saved by `train.ipynb`, reweighted to the
base rate you set. Nothing is retrained when you move a slider.
"""

with gr.Blocks(title="Repilo check") as demo:
    gr.Markdown(
        "# Repilo check\n"
        "Should this olive parcel be sprayed with copper? The model reads one leaf photo "
        "and either leaves the parcel alone, sends it to the agronomist, or recommends "
        "treatment. Move the thresholds to see what each policy costs.",
        elem_id="title")

    with gr.Row():
        t_low = gr.Slider(0, 1, value=review["t_low"], step=0.005,
                          label="Below this, don't treat")
        t_high = gr.Slider(0, 1, value=review["t_high"], step=0.01,
                           label="At or above this, treat")
        base_rate = gr.Slider(0.05, 0.6, value=BASE_RATE, step=0.05,
                              label="Share of visited parcels that really have peacock spot")
    summary = gr.HTML()

    with gr.Tabs():
        with gr.Tab("Check a leaf"):
            with gr.Row():
                image = gr.Image(type="pil", label="Leaf photo", height=340)
                with gr.Column():
                    verdict = gr.HTML()
                    probs = gr.Label(num_top_classes=3, label="Model probabilities")

        with gr.Tab("Thresholds and cost"):
            gr.Markdown("Each dot is a test leaf, placed at the model's probability of peacock "
                        "spot. Filled dots really have peacock spot. The coloured zones are the "
                        "three decisions at the current thresholds.")
            strip = gr.Plot(label="Where the test leaves fall")
            outcome_table = gr.Dataframe(label="Test leaves by true class and decision",
                                         interactive=False)
            curve = gr.Plot(label="Cost against threshold")

        with gr.Tab("Model performance"):
            gr.Markdown(f"Results on the {len(preds)} test images, with peacock spot as the "
                        "positive class. Classes are decided by the most probable label here.")
            with gr.Row():
                gr.Dataframe(per_class.reset_index(names="class"), label="Per-class precision and recall")
                gr.Dataframe(confusion.reset_index(names=""), label="Confusion matrix (rows = true)")
            gr.Dataframe(baselines.reset_index(names="model"), label="Finetuned model against baselines")
            gr.Image(str(RESULTS / "eval_finetuned.png"), label="Confusion matrix and per-class scores")
            gr.Image(str(RESULTS / "test_errors.png"), label="Every test image the model got wrong")

        with gr.Tab("Assumptions"):
            gr.Markdown(ASSUMPTIONS)

    def refresh(lo, hi, br):
        counts, _, _, _ = evaluate_band(lo, hi, br)
        return (summary_html(lo, hi, br), decision_strip(lo, hi),
                counts.reset_index(names="true class"), cost_curve(lo, hi, br))

    controls = [t_low, t_high, base_rate]
    dashboard_outputs = [summary, strip, outcome_table, curve]
    gr.on([s.change for s in controls] + [demo.load], refresh, controls, dashboard_outputs)
    gr.on([image.change, t_low.change, t_high.change], classify,
          [image, t_low, t_high], [probs, verdict])
    demo.load(classify, [image, t_low, t_high], [probs, verdict])

demo.launch(theme=THEME, css=CSS)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
